# מפגש 17 — מחברת הדגמת מורה Audited
Penguins אמיתי, flipper_length_mm ו־body_mass_g, seed=42, K3. דוגמת Accuracy קטנה מומצאת בנפרד. כל פרמטרי ההכנה נלמדים מתוך Train בלבד; Test לא משמשת לבחירת K.

## 1. לפני קוד: 3 נכונות מתוך 4
דוגמה סינתטית: אמת 1,0,0,1; חיזוי 1,0,1,1. השורה השלישית שגויה. Accuracy=3/4=0.75=75%.

In [ ]:
from sklearn.metrics import accuracy_score
y_true_demo = [1, 0, 0, 1]
y_pred_demo = [1, 0, 1, 1]
flags_demo = [a == b for a, b in zip(y_true_demo, y_pred_demo)]
manual_demo = sum(flags_demo) / len(flags_demo)
print("Synthetic demo:", flags_demo, manual_demo, accuracy_score(y_true_demo, y_pred_demo))
assert flags_demo == [True, True, False, True]
assert manual_demo == 0.75
assert accuracy_score(y_true_demo, y_pred_demo) == 0.75

## 2. טוענים Penguins עם Raw CSV
בדקו שמספר העמודות הוא 7, והבחינו בין species (יעד) למדידות גופניות (X).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
print("Source:", df.shape)
display(df.head(3))
display(df.isna().sum())
assert df.shape == (344,7)
assert 'species' in df.columns

## 3. מפצלים X ו־y באותה קריאה
המאפיינים הם flipper_length_mm ו־body_mass_g; מסד העבודה שומר מקור, seed42 ו־stratify=y.

In [ ]:
model_df = df.dropna(subset=["species"]).copy()
features = ["flipper_length_mm","body_mass_g"]
X = model_df[features].copy()
y = model_df["species"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
assert df.shape == (344, 7)
assert X_train.shape == (258, 2)
assert X_test.shape == (86, 2)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert "species" not in X.columns

## 4. משלימים חסרים ומתקננים מ־Train בלבד
משתמשים בממוצעי Train בשתי הקבוצות, ואחרי ההשלמה מתאימים scaler על Train בלבד.

In [ ]:
training_means = X_train.mean()
X_train_ready = X_train.fillna(training_means)
X_test_ready = X_test.fillna(training_means)
scaler = StandardScaler().set_output(transform="pandas")
scaler.fit(X_train_ready)
X_train_scaled = scaler.transform(X_train_ready)
X_test_scaled = scaler.transform(X_test_ready)
display(X_train_scaled.head(3))
assert X_train_scaled.shape == X_train.shape
assert X_test_scaled.shape == X_test.shape
assert X_train_scaled.index.equals(X_train.index)
assert X_test_scaled.index.equals(X_test.index)
assert not X_train_scaled.isna().any().any()
assert not X_test_scaled.isna().any().any()
assert np.allclose(scaler.mean_, X_train_ready.mean().to_numpy())

## 5. מסווג KNN: fit לעומת predict
fit מקבל תכונות ותוויות אימון, predict מקבל **רק** תכונות בדיקה. K3 מוגדר מראש ולא נבחר לפי תוצאת Test.

In [ ]:
model = KNeighborsClassifier(n_neighbors=3, weights="uniform", metric="euclidean")
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)
print("First five predictions:", predictions[:5])
assert len(predictions) == len(y_test)
assert set(predictions).issubset(set(y_train.unique()))
assert model.n_neighbors == 3

## 6. Accuracy ידני מול accuracy_score
לכל שורה אמת, חיזוי, נכון/שגוי. משתמשים באינדקס המקורי של y_test, ולא ב־RangeIndex חדש.

In [ ]:
comparison = pd.DataFrame(
    {"actual": y_test, "predicted": predictions},
    index=y_test.index
)
comparison["correct"] = comparison["actual"] == comparison["predicted"]
correct_count = int(comparison["correct"].sum())
accuracy_hand = correct_count / len(comparison)
accuracy_library = accuracy_score(y_test, predictions)
print("Correct:", correct_count, "/", len(comparison),
      "Accuracy:", accuracy_library, "Percent:", 100 * accuracy_library)
display(comparison.head(10))
assert comparison.index.equals(y_test.index)
assert len(comparison) == 86
assert abs(accuracy_hand - accuracy_library) < 1e-12
assert 0 <= accuracy_library <= 1

## 7. מהן הטעויות בפועל?
אין להמציא שגיאות. מציגים רק טעויות קיימות, עד שלוש, ומחזירים מדידות מאותו אינדקס.

In [ ]:
errors = comparison[comparison["correct"] == False].copy()
print("Errors observed:", len(errors))
display(errors.head(3))
if len(errors) > 0:
    display(X_test.loc[errors.head(3).index])
else:
    print("No errors to display in this split.")
assert correct_count + len(errors) == len(comparison)
assert all(errors["actual"] != errors["predicted"])

## 8. בדיקת טעויות ערכיות נפוצות
בדקו שלא שיניתם מקור, שהתכונות לא כללו יעד ושהפיצול שמר אינדקסים. אין להשתמש בתוצאות Test לבחירת K.

In [ ]:
assert df.shape == (344,7)
assert features == ["flipper_length_mm","body_mass_g"]
assert "species" not in X_train.columns
assert y_test.index.equals(X_test.index)
assert correct_count + len(errors) == 86
assert set(comparison["actual"].unique()).issubset(set(y.unique()))
print("All checks passed. Test was used only for final evaluation.")

## כרטיס יציאה
מה מקבלת fit ומה מקבלת predict? למה אין צורך ב־y_test בזמן predict? האם 0.75 מוכיח הצלחה בטבע? מדוע אין להשוות מודרכת/תלמיד כהוכחה לעדיפות מאפיינים כשהן משתמשות גם ב־seed אחר?